<a href="https://colab.research.google.com/github/dks1532/SNU_BigData_AI_Fintech/blob/main/practice2_solutions.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Practice 2 — Search & Sorting: Merge Sort (Solutions / Teaching Notebook)

**Computing Bootcamp | SNU Graduate School of Data Science**

TA: Reeahn Kim

---

This notebook walks through the merge-sort exercise **step by step**. Plan:

1. Review the big idea — divide, sort, merge.
2. Look at `mergeSortHelp` (given) and check we understand each part.
3. **Build `merge` together**, one phase at a time.
4. **Trace the full example** `[4, 5, 1, 3, 2]` to see how the calls nest.
5. Wrap-up on complexity and common student mistakes.


## 🔰 Jupyter refresher

This notebook is made of **cells** — 📝 markdown (text) and 💻 code. To run a code cell, click it and press **`Shift + Enter`** (runs and moves on) or **`Ctrl/Cmd + Enter`** (runs and stays). The `[ ]` on the left shows a number after the cell has been run — the number tells you the order in which cells were executed.

**⚠️ Run cells top to bottom.** Cells share memory in the order you run them, not the order they appear. If you skip ahead you'll often see `NameError`.

**If something goes wrong:** click ⏹️ **Interrupt** to stop a stuck cell, or `Kernel → Restart & Run All` for a clean slate.


---

## Step 1 — The big idea

Merge sort is a classic **divide-and-conquer** algorithm.

| Phase | What we do |
|---|---|
| **Divide** | Split the list in half. |
| **Sort** | Sort each half — using merge sort again (recursion!). |
| **Merge** | Combine two sorted halves into one sorted list. |

**Base case:** a list of length 1 (or 0) is already sorted — no work needed.

**Why this is fast.** Sorting two halves is *much* cheaper than sorting the whole list at once, and merging two sorted lists is a **linear** operation (just compare the fronts). The tree of recursion has depth `log N`, and we do `O(N)` work at each level → **O(N log N)** total.

**Why we need extra space.** Merging in place is tricky — we need temporary copies of the two halves. That's the O(N) space cost.


## Step 2 — `mergeSortHelp` (already given)

This handles the **Divide + recursive Sort** parts. Read it out loud with the students:

- "If `first == last`, this region is a single element, so it's already sorted — return."
- "Otherwise, find the middle, sort the left half, sort the right half, then merge."

The clever bit: the two recursive calls **cover the whole region** because `first..mid` and `mid+1..last` together tile the range `first..last` with no overlap and no gap.


In [ ]:
def mergeSortHelp(L: list, first: int, last: int) -> None:
    if first == last:               # Base case: single-element region
        return
    else:                            # Recursive case
        # Step 1: Divide it into two sublists
        mid = first + (last - first) // 2
        # Step 2: Sort the two sublists (recursively)
        mergeSortHelp(L, first, mid)
        mergeSortHelp(L, mid + 1, last)
        # Step 3: Merge the two 'sorted' sublists
        merge(L, first, mid, last)


---

## Step 3 — Understanding the `merge` problem

By the time `merge(L, first, mid, last)` is called, we're **guaranteed** that the two halves of `L` are already sorted:

```
         first        mid  mid+1        last
           ↓           ↓    ↓            ↓
    L = [  sorted region 1  |  sorted region 2  ]
           ─────────────────    ─────────────────
                  sub1                sub2
```

Our job is to turn `L[first..last]` into a **single** sorted region.

### The two-pointer trick

Because both halves are already sorted, we don't need to compare every pair of elements. We just look at the **fronts** of both halves and pick the smaller one — that has to be the smallest remaining value overall. Repeat until one side runs out, then dump the rest of the other side.


## Step 4 — Building `merge` piece by piece

### 4a. Initialization

First we **copy** the two halves out of `L` into helper lists `sub1` and `sub2`. This is critical — if we didn't copy them, our writes into `L[k]` would overwrite values we still needed to read from.

We also set up three pointers:
- `i` — index into `sub1`
- `j` — index into `sub2`
- `k` — where in `L` we're writing next (starts at `first`)


### 4b. Phase 1 — both sublists still have elements

```python
while i < len(sub1) and j < len(sub2):
    if sub1[i] < sub2[j]:
        L[k] = sub1[i]
        i += 1
    else:
        L[k] = sub2[j]
        j += 1
    print(L, L[k])
    k += 1
```


### 4c. Phase 2 — copy the leftovers

When Phase 1 exits, **exactly one** of `sub1` or `sub2` still has elements remaining. Since both sublists were already sorted, the leftovers are automatically in order — we just have to copy them into `L`.

```python
while i < len(sub1):
    L[k] = sub1[i]
    i += 1
    print(L, L[k])
    k += 1

while j < len(sub2):
    L[k] = sub2[j]
    j += 1
    print(L, L[k])
    k += 1
```

**Only one of these two loops will actually execute** (the other's condition is false because that sublist is empty). Writing both keeps the code symmetric and easy to reason about.


### 4d. The full `merge` function

In [ ]:
def merge(L: list, first: int, mid: int, last: int) -> None:
    """Merge two adjacent sorted regions of L into one sorted region, in place."""
    # --- Initialization ---
    k = first
    sub1 = L[first : mid + 1]      # copy of the left sorted region
    sub2 = L[mid + 1 : last + 1]   # copy of the right sorted region
    i = j = 0

    # --- Phase 1: both sublists still have elements ---
    while i < len(sub1) and j < len(sub2):
        if sub1[i] < sub2[j]:
            L[k] = sub1[i]
            i += 1
        else:
            L[k] = sub2[j]
            j += 1
        print(L, L[k])
        k += 1

    # --- Phase 2: copy leftovers (only one of these loops runs) ---
    while i < len(sub1):
        L[k] = sub1[i]
        i += 1
        print(L, L[k])
        k += 1

    while j < len(sub2):
        L[k] = sub2[j]
        j += 1
        print(L, L[k])
        k += 1


---

## Step 5 — Trace the full example

Now let's run the given example and match every line of output back to a specific merge call. This is exactly the trace shown on the slides.


In [ ]:
L = [4, 5, 1, 3, 2]
mergeSortHelp(L, 0, 4)
print()
print('Final sorted list:', L)


### Reading the trace

The output shows **12 print statements** — one per element placement. Let's match them to merge calls (see the slide diagrams):

| # | Line | From which `merge` call? | What just happened |
|---|---|---|---|
| 1 | `[4, 5, 1, 3, 2] 4` | `merge(L, 0, 0, 1)` | Compare `sub1=[4]` vs `sub2=[5]` → `4<5`, place 4 |
| 2 | `[4, 5, 1, 3, 2] 5` | `merge(L, 0, 0, 1)` | `sub1` empty, copy leftover `5` |
| 3 | `[1, 5, 1, 3, 2] 1` | `merge(L, 0, 1, 2)` | `sub1=[4,5]` vs `sub2=[1]` → `4≥1`, place 1 |
| 4 | `[1, 4, 1, 3, 2] 4` | `merge(L, 0, 1, 2)` | `sub2` empty, copy leftover `4` |
| 5 | `[1, 4, 5, 3, 2] 5` | `merge(L, 0, 1, 2)` | copy leftover `5` |
| 6 | `[1, 4, 5, 2, 2] 2` | `merge(L, 3, 3, 4)` | `sub1=[3]` vs `sub2=[2]` → `3≥2`, place 2 |
| 7 | `[1, 4, 5, 2, 3] 3` | `merge(L, 3, 3, 4)` | copy leftover `3` |
| 8 | `[1, 4, 5, 2, 3] 1` | `merge(L, 0, 2, 4)` | `sub1=[1,4,5]` vs `sub2=[2,3]` → `1<2`, place 1 |
| 9 | `[1, 2, 5, 2, 3] 2` | `merge(L, 0, 2, 4)` | `4≥2`, place 2 |
| 10 | `[1, 2, 3, 2, 3] 3` | `merge(L, 0, 2, 4)` | `4≥3`, place 3 |
| 11 | `[1, 2, 3, 4, 3] 4` | `merge(L, 0, 2, 4)` | `sub2` empty, copy leftover `4` |
| 12 | `[1, 2, 3, 4, 5] 5` | `merge(L, 0, 2, 4)` | copy leftover `5` |

**Key insight to point out:** the last four writes (lines 8–12) all happen inside the **final** `merge(L, 0, 2, 4)` call. That's the merge at the *root* of the recursion tree — the one that stitches the two fully-sorted halves back together. Notice how at that point, the left half `[1, 4, 5]` and the right half `[2, 3]` are each already sorted — that's the invariant merge sort relies on.


### Visualizing the recursion tree

The recursion produces this call structure (matches the slide's tree diagrams):

```
                mergeSortHelp(L, 0, 4)                mid=2
                /                    \
     mergeSortHelp(L, 0, 2)      mergeSortHelp(L, 3, 4)
        mid=1                        mid=3
       /        \                   /        \
  MSH(L,0,1)  MSH(L,2,2)       MSH(L,3,3)  MSH(L,4,4)
  mid=0        (base)           (base)      (base)
   /    \
 MSH(L,0,0)  MSH(L,1,1)
  (base)      (base)
```

Merges happen **on the way back up** — leaves first, root last. That's why the 4-element final merge appears at the very end of the output.


---

## Step 6 — Sanity checks on more inputs

Let's confirm the implementation handles the tricky cases correctly.


In [ ]:
# Already sorted — merge sort should still work, just makes lots of trivial comparisons
L1 = [1, 2, 3, 4, 5]
mergeSortHelp(L1, 0, 4)
print('Already sorted →', L1)
print()


In [ ]:
# Reverse sorted — every comparison goes the same way
L2 = [5, 4, 3, 2, 1]
mergeSortHelp(L2, 0, 4)
print('Reverse sorted →', L2)
print()


In [ ]:
# With duplicates — good place to note stability (equal keys keep original order)
L3 = [3, 1, 4, 1, 5, 9, 2, 6, 5, 3, 5]
mergeSortHelp(L3, 0, len(L3) - 1)
print('With duplicates →', L3)


---